# TADs Analysis

Differential analysis of TAD borders between schizophrenia and healthy control samples, including GO enrichment analysis.

In [ ]:
import pandas as pd
import numpy as np
import bioframe as bf
import gseapy as gp
from gseapy import barplot, dotplot
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
merged_bed = pd.read_pickle("./tads_additional_files/merged_bed_with_tads_clustering_neurons.pickle")
merged_bed.cluster.nunique()

6856

In [79]:
merged_bed['group'] = merged_bed.source.apply(lambda x: "SZ" if "SZ" in x else "HC")
(set(merged_bed["group"])) 

group_sets = merged_bed.groupby('cluster')['group'].agg(set)
unique_borders_sz = group_sets[group_sets.apply(lambda g: 'SZ' in g and 'HC' not in g)].index.to_list()
unique_borders_hc = group_sets[group_sets.apply(lambda g: 'HC' in g and 'SZ' not in g)].index.to_list()
assert not set(unique_borders_hc) & set(unique_borders_sz)
print(len(unique_borders_hc), len(unique_borders_sz))

{'HC', 'SZ'}

## Identify Differential Borders

In [59]:
def get_list_of_up_down_borders(merged_bed, p_col = '_p_value', p_value_cutoff=0.05):
    modes = ['HC_vs_SZ']
    print(modes)
    assert len(modes) == 1
    mode = modes[0]
    
    print("____"*14)
    diff_borders = {}
    print(mode)
    col_tstat = mode+"_mu_stat"
    col_pvalue = mode+p_col
    
    deduplicated_clusters = merged_bed[["cluster", col_tstat]].drop_duplicates()[col_tstat].tolist()
    lower_quantile = np.median(deduplicated_clusters)
    upper_quantile = np.median(deduplicated_clusters)
    # print(f"Q1, Q3: {lower_quantile}, {upper_quantile}") 
    
    for sign in ["up", "down"]:
        print(sign)
        if sign == "up":
            filtered_df = merged_bed.loc[(merged_bed[col_tstat] > upper_quantile )&(merged_bed[col_pvalue] <= p_value_cutoff)]
        else:
            filtered_df = merged_bed.loc[(merged_bed[col_tstat] < lower_quantile )&(merged_bed[col_pvalue] <= p_value_cutoff)]
        print(filtered_df.cluster.nunique())
        print(f"Number of diff borders that are {sign} in comparison - {filtered_df.cluster.nunique()}")
        diff_borders[sign] = filtered_df
    all_other_boders = merged_bed[~merged_bed.cluster.isin(diff_borders["up"].cluster.tolist()+ diff_borders["down"].cluster.tolist())]
    diff_borders['not_changed'] = all_other_boders
    return diff_borders 

In [265]:
diff_borders = get_list_of_up_down_borders(merged_bed)

['HC_vs_SZ']
________________________________________________________
HC_vs_SZ
up
90
Number of diff borders that are up in comparison - 90
down
120
Number of diff borders that are down in comparison - 120


In [267]:
diff_borders['up']

,chrom,start,end,boundary_strength_150000,source,cluster,HC_vs_SZ_mu_stat,HC_vs_SZ_p_value,group
305,chr1,17055000,17070000,1.284162,HC-2Mplus_sampled_drop_diag_15res_150wind.csv,chr1_31,36.0,0.002165,HC
306,chr1,17055000,17070000,1.353653,HC-318plus_sampled_drop_diag_15res_150wind.csv,chr1_31,36.0,0.002165,HC
307,chr1,17070000,17085000,1.413820,HC-3Mplus_sampled_drop_diag_15res_150wind.csv,chr1_31,36.0,0.002165,HC
308,chr1,17055000,17070000,1.204452,HC-91plus_sampled_drop_diag_15res_150wind.csv,chr1_31,36.0,0.002165,HC
309,chr1,17055000,17070000,1.432912,HC24plus_sampled_drop_diag_15res_150wind.csv,chr1_31,36.0,0.002165,HC
...,...,...,...,...,...,...,...,...,...
60740,chrX,79485000,79500000,0.704694,HC24plus_sampled_drop_diag_15res_150wind.csv,chrX_111,30.0,0.028441,HC
61831,chrX,121695000,121710000,0.422143,HC-318plus_sampled_drop_diag_15res_150wind.csv,chrX_250,30.0,0.028441,HC
61832,chrX,121695000,121710000,0.357222,HC-91plus_sampled_drop_diag_15res_150wind.csv,chrX_250,30.0,0.028441,HC
61833,chrX,121665000,121680000,0.373401,HC24plus_sampled_drop_diag_15res_150wind.csv,chrX_250,30.0,0.028441,HC


## Run GO for Genes in Differential Borders

In [62]:

def get_genes_lists(df):
    genes_in_up_list = df.loc[
        df["gene_biotype_anno"] == "protein_coding"].dropna()[
        ["cluster", "gene_biotype_anno", "gene_name_anno"]].drop_duplicates().gene_name_anno
    genes_in_up_list = list(set(genes_in_up_list))
    return genes_in_up_list

def run_go(genes_in_important_clusters, genes_in_all_clusters, add_pathways=True, name= None):
    if name:
        print(f"Dataset - {name}")
    genes = genes_in_important_clusters[genes_in_important_clusters.gene_biotype_anno == "protein_coding"].gene_name_anno.dropna().unique().tolist()
    background = genes_in_all_clusters[genes_in_all_clusters.gene_biotype_anno == "protein_coding"].gene_name_anno.dropna().unique().tolist()
    print(f"In gene_list - {len(genes)} genes, in the background - {len(background)}")
    gene_set = ['GO_Biological_Process_2021',
                'GO_Cellular_Component_2021',
                'GO_Molecular_Function_2021']
    if add_pathways:
        gene_set+=['MSigDB_Hallmark_2020','KEGG_2021_Human',  "WikiPathway_2023_Human"]
    enr = gp.enrichr(gene_list=genes,
                     background=background,
                     gene_sets=gene_set,
                     organism='Human',
                     outdir=None, 
                    )
    return enr

def get_dot_plot(enr):
    if enr.results.query('`Adjusted P-value` < 0.05').shape[0] > 0:
        cnt = 0
        for i, term in enumerate(enr.results.query('`Adjusted P-value` < 0.05').sort_values('Adjusted P-value').Term.tolist()):
            print(f"{i+1}. {term}")
            cnt+=1
        if cnt <= 4:
            height = 3
            width = 5
        elif 4< cnt < 9:
            height = 8
            width = 7

        elif 9<= cnt < 11:
            height = 12
            width = 9
        else:
            height = 16
            width = 9
        ax = dotplot(enr.results,
                  column="Adjusted P-value",
                  x='Gene_set', # set x axis, so you could do a multi-sample/library comparsion
                  size=3,
                  top_term=10,
                  figsize=(width,height),
                  title = "GO",
                  xticklabels_rot=45, # rotate xtick labels
                  show_ring=True, # set to False to revmove outer ring
                  marker='o',
                 )
        plt.show()
    else:
        print('No enriched terms at Adjusted P-value < 0.05')


def get_genes_in_borders(df_init, promoters , factor_slop = 1):
    df = df_init.copy()
    df.start =df.start - 15000 *factor_slop
    df.end =df.end + 15000  *factor_slop
    df.start = df.start.astype(int)
    df.end = df.end.astype(int)
    genes_in_important_clusters = bf.overlap(df, promoters, how='left', suffixes=('','_anno'))
    print(f"All genes - {genes_in_important_clusters.gene_name_anno.nunique()}")
    pc_genes_in_important_clusters = genes_in_important_clusters[genes_in_important_clusters.gene_biotype_anno == "protein_coding"].reset_index(drop=True)    
    print(f'Protein - codings genes - {pc_genes_in_important_clusters.gene_name_anno.nunique()}')
    pc_genes_in_important_clusters = get_genes_lists(pc_genes_in_important_clusters)
    return genes_in_important_clusters, pc_genes_in_important_clusters


In [268]:
gtf_genes = pd.read_feather('/tank/projects/diana_hic/sz_project2024/2.4.loops_analysis_extended/genes_homo_sapience.feather')
promoters = gtf_genes[['chrom', 'promoter_start', 'promoter_end', 'gene_name', 'gene_biotype',
                                   'gene_id' ]]
promoters.columns = ['chrom', 'start', 'end', 'gene_name', 'gene_biotype',
                                   'gene_id']

In [269]:
genes_up, pc_genes_up = get_genes_in_borders(diff_borders["up"], promoters )
genes_down, pc_genes_down = get_genes_in_borders(diff_borders["down"], promoters )
genes_all, pc_genes_all = get_genes_in_borders(merged_bed, promoters )
    

All genes - 153
Protein - codings genes - 74
All genes - 198
Protein - codings genes - 121
All genes - 9988
Protein - codings genes - 5706


In [65]:
go_up = run_go(genes_up,genes_all )
get_dot_plot(go_up)

No enriched terms at Adjusted P-value < 0.05


In [66]:
go_down = run_go(genes_down,genes_all )
get_dot_plot(go_down)

No enriched terms at Adjusted P-value < 0.05


## Borders Missing in One Group

In [96]:
merged_bed['group'] = merged_bed.source.apply(lambda x: "SZ" if "SZ" in x else "HC")
(set(merged_bed["group"])) 

group_sets = merged_bed.groupby('cluster')['group'].agg(set)
unique_borders_sz = group_sets[group_sets.apply(lambda g: 'SZ' in g and 'HC' not in g)].index.to_list()
unique_borders_hc = group_sets[group_sets.apply(lambda g: 'HC' in g and 'SZ' not in g)].index.to_list()
assert not set(unique_borders_hc) & set(unique_borders_sz)
print(len(unique_borders_hc), len(unique_borders_sz))

77 62


In [98]:
genes_only_sz, pc_genes_only_sz = get_genes_in_borders(merged_bed[merged_bed.cluster.isin(unique_borders_sz)].reset_index(drop=True), promoters )
genes_only_hc, pc_genes_only_hc = get_genes_in_borders(merged_bed[merged_bed.cluster.isin(unique_borders_hc)].reset_index(drop=True), promoters )
genes_all, pc_genes_all = get_genes_in_borders(merged_bed, promoters )
    

All genes - 41
Protein - codings genes - 27
All genes - 77
Protein - codings genes - 39
All genes - 9988
Protein - codings genes - 5706


In [99]:
go_up = run_go(genes_only_sz,genes_all )
get_dot_plot(go_up)

In gene_list - 27 genes, in the background - 5706
No enriched terms at Adjusted P-value < 0.05


In [103]:
set(genes_only_sz.gene_name_anno.tolist())

{'ABCB5',
 'ADAM29',
 'BBS12',
 'CCNA1',
 'CETN4P',
 'DOCK5',
 'EIF3E',
 'ELOCP3',
 'EMC2',
 'EPB41',
 'ERBB3',
 'ESYT1',
 'F13A1',
 'FAAH',
 'FAF2P1',
 'FAM135B',
 'GLRA3',
 'GPR119',
 'GRPR',
 'LEO1',
 'LINC01080',
 'LINC02109',
 'MAGEB17',
 'MAPK6',
 'MIR3936',
 'MIR3936HG',
 'MIR769',
 'MIR8053',
 'MTCYBP38',
 'NGRNP1',
 None,
 'PA2G4',
 'PGBD4P6',
 'PGLYRP1',
 'RGS2',
 'RNU6-90P',
 'RPL41',
 'SLC22A5',
 'SLC25A14',
 'UBXN4',
 'Y_RNA',
 'ZC3H10'}

In [100]:
go_down = run_go(genes_only_hc,genes_all )
get_dot_plot(go_down)

In gene_list - 39 genes, in the background - 5706
No enriched terms at Adjusted P-value < 0.05
